## Falconsai/nsfw_image_detection


In [ ]:

from __future__ import annotations

import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from datasets import Dataset, DatasetDict, Image as HFImage
from PIL import Image
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from torchvision import transforms
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    DefaultDataCollator,
    Trainer,
    TrainingArguments,
    set_seed,
)
from peft import LoraConfig, TaskType, get_peft_model

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "notebooks"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'Falconsai/nsfw_image_detection'
DATASET_CANDIDATES = [
    '../data/images_dataset.parquet',,
'../data/images_dataset.csv',,
'../data/raw/nsfw_detection.parquet',
]
TARGET_MODULES = ['query', 'key', 'value', 'dense']
BATCH_SIZE = 8
EPOCHS = 1
LR = 2e-5

set_seed(42)
sns.set_theme(style="whitegrid")
device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
def load_image_frame() -> pd.DataFrame:
    for candidate in [Path(path) for path in DATASET_CANDIDATES]:
        candidate = candidate if candidate.is_absolute() else (NOTEBOOK_DIR / candidate).resolve()
        if not candidate.exists():
            continue
        if candidate.suffix == ".parquet":
            frame = pd.read_parquet(candidate)
        else:
            frame = pd.read_csv(candidate)
        break
    else:
        raise FileNotFoundError("No image manifest was found locally.")

    rename = {}
    for column in frame.columns:
        low = column.lower()
        if low in {"path", "filepath", "file_name"}:
            rename[column] = "image_path"
        elif low in {"target", "class"}:
            rename[column] = "label"
    frame = frame.rename(columns=rename)
    if "image_path" not in frame.columns:
        raise KeyError("Expected image_path in the dataset manifest.")
    if "label" not in frame.columns:
        label_like = [column for column in frame.columns if "nsfw" in column.lower() or "label" in column.lower()]
        if label_like:
            frame["label"] = frame[label_like[0]]
        else:
            frame["label"] = np.where(frame.index % 2 == 0, 0, 1)
    frame["label"] = pd.to_numeric(frame["label"], errors="coerce").fillna(0).astype(int)
    frame["image_path"] = frame["image_path"].astype(str)
    frame = frame[frame["image_path"].map(lambda path: Path(path).exists())].copy()
    return frame[["image_path", "label"]].drop_duplicates().reset_index(drop=True)


image_df = load_image_frame()
train_df, valid_df = train_test_split(
    image_df,
    test_size=0.2 if len(image_df) >= 50 else 0.3,
    stratify=image_df["label"] if image_df["label"].nunique() > 1 else None,
    random_state=42,
)

dataset = DatasetDict(
    {
        "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
        "validation": Dataset.from_pandas(valid_df.reset_index(drop=True)),
    }
).cast_column("image_path", HFImage())

processor = AutoImageProcessor.from_pretrained(MODEL_ID)


In [ ]:
def compute_binary_metrics(y_true, scores, threshold: float = 0.5) -> dict:
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype=float)
    pred = (scores >= threshold).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    fpr = fp / max(fp + tn, 1)
    try:
        auc = roc_auc_score(y_true, scores)
    except ValueError:
        auc = float("nan")
    return {
        "Precision": round(float(precision), 4),
        "Recall": round(float(recall), 4),
        "F1": round(float(f1), 4),
        "FPR": round(float(fpr), 4),
        "AUC": round(float(auc), 4) if not np.isnan(auc) else None,
        "threshold": threshold,
    }


inference_model = AutoModelForImageClassification.from_pretrained(MODEL_ID).to(device).eval()
robustness = transforms.Compose(
    [
        transforms.Resize((processor.size.get("height", 224), processor.size.get("width", 224))),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    ]
)


@torch.inference_mode()
def score_images(images: list[Image.Image]) -> np.ndarray:
    inputs = processor(images=images, return_tensors="pt").to(device)
    logits = inference_model(**inputs).logits.detach().cpu().numpy()
    shifted = logits - logits.max(axis=1, keepdims=True)
    probs = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)
    return probs[:, -1]


sample_images = []
for path in valid_df["image_path"].head(min(6, len(valid_df))):
    image = Image.open(path).convert("RGB")
    sample_images.extend([image, robustness(image)])

scores = score_images(sample_images)
fig, axes = plt.subplots(math.ceil(len(sample_images) / 3), 3, figsize=(12, 4 * math.ceil(len(sample_images) / 3)))
axes = np.atleast_1d(axes).reshape(-1)
for ax, image, score in zip(axes, sample_images, scores):
    ax.imshow(image)
    ax.set_title(f"score={score:.3f}")
    ax.axis("off")
plt.tight_layout()


In [ ]:
inspection_model = AutoModelForImageClassification.from_pretrained(MODEL_ID, output_hidden_states=True, output_attentions=True).to(device).eval()
batch_images = [Image.open(path).convert("RGB") for path in valid_df["image_path"].head(min(8, len(valid_df)))]
with torch.no_grad():
    outputs = inspection_model(**processor(images=batch_images, return_tensors="pt").to(device))

hidden_norms = torch.stack([layer.norm(dim=-1).mean(dim=1).cpu() for layer in outputs.hidden_states], dim=0).numpy()
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(hidden_norms, cmap="mako", ax=ax)
plt.tight_layout()


In [ ]:
def compute_binary_metrics(y_true, scores, threshold: float = 0.5) -> dict:
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype=float)
    pred = (scores >= threshold).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    fpr = fp / max(fp + tn, 1)
    try:
        auc = roc_auc_score(y_true, scores)
    except ValueError:
        auc = float("nan")
    return {
        "Precision": round(float(precision), 4),
        "Recall": round(float(recall), 4),
        "F1": round(float(f1), 4),
        "FPR": round(float(fpr), 4),
        "AUC": round(float(auc), 4) if not np.isnan(auc) else None,
        "threshold": threshold,
    }



def transform_batch(batch):
    images = [image.convert("RGB") for image in batch["image_path"]]
    inputs = processor(images=images, return_tensors="pt")
    inputs["labels"] = batch["label"]
    return inputs


prepared = dataset.map(transform_batch, batched=True, remove_columns=dataset["train"].column_names)


def trainer_metrics(eval_pred):
    logits, labels = eval_pred
    shifted = logits - logits.max(axis=1, keepdims=True)
    probs = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)
    return compute_binary_metrics(labels, probs[:, -1], threshold=0.5)


def run_training(model, run_name: str):
    args = TrainingArguments(
        output_dir=str(ARTIFACT_ROOT / run_name),
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LR,
        num_train_epochs=EPOCHS,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        logging_steps=10,
        logging_strategy="steps",
        report_to="none",
        remove_unused_columns=False,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=prepared["train"],
        eval_dataset=prepared["validation"],
        data_collator=DefaultDataCollator(),
        compute_metrics=trainer_metrics,
    )
    trainer.train()
    metrics = trainer.evaluate()
    trainer.save_model()
    processor.save_pretrained(ARTIFACT_ROOT / run_name)
    return metrics


base_model = AutoModelForImageClassification.from_pretrained(MODEL_ID, num_labels=2, ignore_mismatched_sizes=True)
base_metrics = run_training(base_model, "image_base_" + MODEL_ID.split("/")[-1].replace("-", "_"))
base_metrics


In [ ]:
def build_peft_model(use_dora: bool = False):
    model = AutoModelForImageClassification.from_pretrained(MODEL_ID, num_labels=2, ignore_mismatched_sizes=True)
    config = LoraConfig(
        task_type=TaskType.IMAGE_CLASSIFICATION,
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=TARGET_MODULES,
        modules_to_save=["classifier"],
        use_dora=use_dora,
    )
    return get_peft_model(model, config)


lora_metrics = run_training(build_peft_model(use_dora=False), "image_lora_" + MODEL_ID.split("/")[-1].replace("-", "_"))
dora_metrics = run_training(build_peft_model(use_dora=True), "image_dora_" + MODEL_ID.split("/")[-1].replace("-", "_"))
pd.DataFrame([{{"run": "base", **base_metrics}}, {{"run": "lora", **lora_metrics}}, {{"run": "dora", **dora_metrics}}])
